# Task 12: ΔΔG 消融实验 — ESM2 ΔΔG vs 结构 ΔΔG vs 全部联合

**问题**: ESM2 零样本 ΔΔG (ddg_esm2) 和三种结构 ΔΔG (ddg_struct + ddg_rasp + ddg_foldx) 是否互补？

| 配置 | 特征 | 维度 |
|---|---|---|
| **v3 + PCA(50) + ΔΔG** | 50PC + 7struct + ddg_esm2 | 58 |
| **v3 + PCA(50) + all_3** | 50PC + 7struct + ddg_struct + ddg_rasp + ddg_foldx | 60 |
| **v3 + PCA(50) + ΔΔG + all_3** | 50PC + 7struct + 4×ddg | 61 |

**对照**: 同时跑 v3 + PCA(50) 作为基线。

In [6]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

# ===== 加载特征矩阵 =====
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")

ID_COLS = ["KEY", "Gene", "reloc_v3"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]
esm_cols = [c for c in feat_cols if c.startswith("esm_")]
STRUCT_NAMES = ["plddt","sasa","rsa","ss_helix","ss_strand","ss_coil",
                "delta_hydrophobicity"]

X_full_arr = df_feat[feat_cols].values.astype(np.float32)
esm_idx = [feat_cols.index(c) for c in esm_cols]
struct_idx = [feat_cols.index(c) for c in STRUCT_NAMES if c in feat_cols]
X_esm = X_full_arr[:, esm_idx]
X_struct = X_full_arr[:, struct_idx]

y_bin = (df_feat["reloc_v3"].values > 0).astype(int)
groups = df_feat["Gene"].values
full_keys = (df_main["Gene"] + "||" + df_main["Mutation_used"]).tolist()

print(f"n={len(y_bin)}, pos={int(y_bin.sum())}, base_rate={y_bin.sum()/len(y_bin):.4f}")

# ===== 加载全部 4 种 ddg =====
ddg_sources = {}
ddg_files = {
    "ddg_esm2":   "ddg_esm2.csv",
    "ddg_struct": "ddg_struct.csv",
    "ddg_rasp":   "ddg_rasp.csv",
    "ddg_foldx":  "ddg_foldx.csv",
}

for name, fname in ddg_files.items():
    path = BASE_PATH + fname
    df_d = pd.read_csv(path)
    # 每个 ddg csv 的列名可能不同，取对应列
    if name == "ddg_esm2":
        dcol = "ddg_esm2"
    elif name == "ddg_struct":
        dcol = "ddg_struct"
    else:
        dcol = name  # ddg_rasp, ddg_foldx
    ddg_map = dict(zip(df_d["KEY"], df_d[dcol]))
    arr = np.array([ddg_map.get(k, np.nan) for k in full_keys], dtype=np.float32)
    n_valid = np.isfinite(arr).sum()
    ddg_sources[name] = arr.reshape(-1, 1)
    print(f"  {name:12s}: {n_valid}/{len(arr)} 有效")

print(f"\n已加载 {len(ddg_sources)} 种 ddg")

n=2179, pos=235, base_rate=0.1078
  ddg_esm2    : 2179/2179 有效
  ddg_struct  : 2168/2179 有效
  ddg_rasp    : 2168/2179 有效
  ddg_foldx   : 2166/2179 有效

已加载 4 种 ddg


## 12a. 同折 CV: 四种配置对比

- **PCA(50) 基线**: 50PC + 8struct
- **+ΔΔG**: + ddg_esm2
- **+all_3**: + ddg_struct + ddg_rasp + ddg_foldx
- **+ΔΔG+all_3**: + 全部 4 种 ddg

In [2]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
N_COMP = 50

CONFIGS = ["PCA", "PCA+ddg_esm2", "PCA+all_3", "PCA+ddg_esm2+all_3"]
oof = {k: np.zeros(len(y_bin), dtype=np.float32) for k in CONFIGS}

for fold, (tr_idx, te_idx) in enumerate(cv.split(X_full_arr, y_bin, groups)):
    y_tr = y_bin[tr_idx]; y_te = y_bin[te_idx]

    # --- ESM2 → PCA ---
    imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
    Xe_tr = scl_e.fit_transform(imp_e.fit_transform(X_esm[tr_idx])).astype(np.float32)
    Xe_te = scl_e.transform(imp_e.transform(X_esm[te_idx])).astype(np.float32)
    pca = PCA(n_components=N_COMP, random_state=42)
    Xe_tr_pca = pca.fit_transform(Xe_tr).astype(np.float32)
    Xe_te_pca = pca.transform(Xe_te).astype(np.float32)

    # --- Struct ---
    imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
    Xs_tr = scl_s.fit_transform(imp_s.fit_transform(X_struct[tr_idx])).astype(np.float32)
    Xs_te = scl_s.transform(imp_s.transform(X_struct[te_idx])).astype(np.float32)

    X_tr_base = np.hstack([Xe_tr_pca, Xs_tr]).astype(np.float32)
    X_te_base = np.hstack([Xe_te_pca, Xs_te]).astype(np.float32)

    sw = compute_sample_weight("balanced", y_tr)

    def fit_pred(X_tr, X_te):
        m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.5,
                          objective="binary:logistic", eval_metric="aucpr",
                          random_state=42, n_jobs=-1, tree_method="hist")
        m.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
        return m.predict_proba(X_te)[:, 1]

    # ---- PCA 基线 ----
    oof["PCA"][te_idx] = fit_pred(X_tr_base, X_te_base)

    # ---- +ddg_esm2 ----
    imp_d1 = SimpleImputer(strategy="median")
    d_esm2_tr = imp_d1.fit_transform(ddg_sources["ddg_esm2"][tr_idx]).astype(np.float32)
    d_esm2_te = imp_d1.transform(ddg_sources["ddg_esm2"][te_idx]).astype(np.float32)
    oof["PCA+ddg_esm2"][te_idx] = fit_pred(
        np.hstack([X_tr_base, d_esm2_tr]).astype(np.float32),
        np.hstack([X_te_base, d_esm2_te]).astype(np.float32))

    # ---- +all_3 (struct + rasp + foldx) ----
    all3_parts_tr = []; all3_parts_te = []
    for name in ["ddg_struct", "ddg_rasp", "ddg_foldx"]:
        imp_d = SimpleImputer(strategy="median")
        all3_parts_tr.append(imp_d.fit_transform(ddg_sources[name][tr_idx]).astype(np.float32))
        all3_parts_te.append(imp_d.transform(ddg_sources[name][te_idx]).astype(np.float32))
    oof["PCA+all_3"][te_idx] = fit_pred(
        np.hstack([X_tr_base] + all3_parts_tr).astype(np.float32),
        np.hstack([X_te_base] + all3_parts_te).astype(np.float32))

    # ---- +ddg_esm2+all_3 (全部 4 种) ----
    all4_parts_tr = [d_esm2_tr] + all3_parts_tr
    all4_parts_te = [d_esm2_te] + all3_parts_te
    oof["PCA+ddg_esm2+all_3"][te_idx] = fit_pred(
        np.hstack([X_tr_base] + all4_parts_tr).astype(np.float32),
        np.hstack([X_te_base] + all4_parts_te).astype(np.float32))

    vals = {k: roc_auc_score(y_te, oof[k][te_idx]) for k in CONFIGS}
    print(f"  Fold {fold}: " + "  ".join([f"{k}={v:.4f}" for k, v in vals.items()]))

  Fold 0: PCA=0.6685  PCA+ddg_esm2=0.6705  PCA+all_3=0.6827  PCA+ddg_esm2+all_3=0.7097
  Fold 1: PCA=0.6246  PCA+ddg_esm2=0.6506  PCA+all_3=0.6185  PCA+ddg_esm2+all_3=0.6535
  Fold 2: PCA=0.5527  PCA+ddg_esm2=0.5225  PCA+all_3=0.5548  PCA+ddg_esm2+all_3=0.5458
  Fold 3: PCA=0.6048  PCA+ddg_esm2=0.6398  PCA+all_3=0.6013  PCA+ddg_esm2+all_3=0.6389
  Fold 4: PCA=0.6204  PCA+ddg_esm2=0.5919  PCA+all_3=0.6223  PCA+ddg_esm2+all_3=0.6005


## 12b. 汇总对比

In [7]:
br = float(y_bin.sum() / len(y_bin))
results = {k: {"auroc": roc_auc_score(y_bin, oof[k]),
                "auprc": average_precision_score(y_bin, oof[k])}
           for k in CONFIGS}
auroc_pca = results["PCA"]["auroc"]

print(f"\n{'='*85}")
print(f"  Task 12: ddG 消融实验 (n={len(y_bin)}, pos={int(y_bin.sum())}, base_rate={br:.4f})")
print(f"  {'配置':<40s} {'AUROC':>8s} {'AUPRC':>8s} {'ΔAUROC':>12s} {'维度':>6s}")
print(f"  {'-'*75}")

dims = {"PCA": 57, "PCA+ddg_esm2": 58, "PCA+all_3": 60, "PCA+ddg_esm2+all_3": 61}
labels = {"PCA": "PCA(50) 基线",
          "PCA+ddg_esm2": "PCA(50) + ddG (ESM2 零样本)",
          "PCA+all_3": "PCA(50) + all_3 (结构×3)",
          "PCA+ddg_esm2+all_3": "PCA(50) + ddG + all_3 (全部×4)"}

for k in CONFIGS:
    r = results[k]
    d = r["auroc"] - auroc_pca
    m = " +" if d > 0.005 else ("" if d > -0.005 else " -")
    print(f"  {labels[k]:<40s} {r['auroc']:>8.4f} {r['auprc']:>8.4f} {d:>+12.4f}{m} {dims[k]:>5d}")

print(f"{'='*85}")

# ===== 增益分解 =====
print(f"\n增益分解:")
d_esm2  = results["PCA+ddg_esm2"]["auroc"] - auroc_pca
d_all3  = results["PCA+all_3"]["auroc"] - auroc_pca
d_all4  = results["PCA+ddg_esm2+all_3"]["auroc"] - auroc_pca

print(f"  ddg_esm2 单独增益:     {d_esm2:+.4f}")
print(f"  all_3 单独增益:         {d_all3:+.4f}")
print(f"  二者相加 (预期):        {d_esm2 + d_all3:+.4f}")
print(f"  全部联合实际增益:       {d_all4:+.4f}")
synergy = d_all4 - (d_esm2 + d_all3)
print(f"  协同/冗余:              {synergy:+.4f}  {'(协同!)' if synergy > 0.003 else ('(冗余)' if synergy < -0.003 else '(≈独立)')}")

# ===== 最佳配置 =====
best_auroc = max(r["auroc"] for r in results.values())
best_name = [k for k, r in results.items() if r["auroc"] == best_auroc][0]
print(f"\n最佳配置: {labels[best_name]} (AUROC={best_auroc:.4f})")
print(f"AlphaMissense baseline: 0.6374, gap={0.6374 - best_auroc:.4f}")


  Task 12: ddG 消融实验 (n=2179, pos=235, base_rate=0.1078)
  配置                                          AUROC    AUPRC       ΔAUROC     维度
  ---------------------------------------------------------------------------
  PCA(50) 基线                                 0.6144   0.1558      +0.0000    57
  PCA(50) + ddG (ESM2 零样本)                   0.6187   0.1700      +0.0043    58
  PCA(50) + all_3 (结构×3)                     0.6155   0.1694      +0.0011    60
  PCA(50) + ddG + all_3 (全部×4)               0.6330   0.1719      +0.0186 +    61

增益分解:
  ddg_esm2 单独增益:     +0.0043
  all_3 单独增益:         +0.0011
  二者相加 (预期):        +0.0054
  全部联合实际增益:       +0.0186
  协同/冗余:              +0.0132  (协同!)

最佳配置: PCA(50) + ddG + all_3 (全部×4) (AUROC=0.6330)
AlphaMissense baseline: 0.6374, gap=0.0044


## 12c. 特征重要性 (全部联合, fit on all data)

In [8]:
print("\n--- 特征重要性 PCA(50)+ddG+all_3 (fit on all data, 仅用于排名) ---")

# PCA on all data
imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
Xe_all = scl_e.fit_transform(imp_e.fit_transform(X_esm)).astype(np.float32)
pca_all = PCA(n_components=N_COMP, random_state=42)
Xe_pca_all = pca_all.fit_transform(Xe_all).astype(np.float32)

# Struct
imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
Xs_all = scl_s.fit_transform(imp_s.fit_transform(X_struct)).astype(np.float32)

# All 4 ddgs
parts = [Xe_pca_all, Xs_all]
ddg_order = []
for name in ["ddg_esm2", "ddg_struct", "ddg_rasp", "ddg_foldx"]:
    imp_d = SimpleImputer(strategy="median")
    d_all = imp_d.fit_transform(ddg_sources[name]).astype(np.float32)
    parts.append(d_all)
    ddg_order.append(name)

X_all = np.hstack(parts).astype(np.float32)
all_names = [f"PC{i+1}" for i in range(N_COMP)] + STRUCT_NAMES + ddg_order

sw_ratio = (y_bin==0).sum() / max((y_bin==1).sum(), 1)
xgb_fi = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.5,
                       scale_pos_weight=sw_ratio,
                       objective="binary:logistic", eval_metric="aucpr",
                       random_state=42, n_jobs=-1, tree_method="hist")
xgb_fi.fit(X_all, y_bin, verbose=False)

imps = xgb_fi.feature_importances_
idx_sorted = np.argsort(imps)[::-1]

print("Top-20 特征:")
for rank, i in enumerate(idx_sorted[:20]):
    val = imps[i]; bar = "\u2588" * int(val * 2000)
    print(f"  {rank+1:2d}. {all_names[i]:30s}  {val:.5f}  {bar}")

print(f"\n四种 ddg 排名 (共 {len(all_names)} 个特征):")
for name in ddg_order:
    i = all_names.index(name)
    rank = int(np.where(idx_sorted == i)[0][0]) + 1
    marker = " \u2605 top-5!" if rank <= 5 else (" \u2605 top-10!" if rank <= 10 else (" \u2605 top-20!" if rank <= 20 else ""))
    print(f"  {name:15s} 排名={rank:3d}/{len(all_names)}  重要性={imps[i]:.5f}{marker}")

print(f"\n结构特征排名:")
for name in STRUCT_NAMES:
    i = all_names.index(name)
    rank = int(np.where(idx_sorted == i)[0][0]) + 1
    marker = " \u2605 top-10!" if rank <= 10 else (" \u2605 top-20!" if rank <= 20 else "")
    print(f"  {name:25s} 排名={rank:3d}/{len(all_names)}  重要性={imps[i]:.5f}{marker}")


--- 特征重要性 PCA(50)+ddG+all_3 (fit on all data, 仅用于排名) ---
Top-20 特征:
   1. ddg_esm2                        0.03955  ███████████████████████████████████████████████████████████████████████████████
   2. ss_coil                         0.02481  █████████████████████████████████████████████████
   3. ddg_rasp                        0.02418  ████████████████████████████████████████████████
   4. PC36                            0.02228  ████████████████████████████████████████████
   5. plddt                           0.02142  ██████████████████████████████████████████
   6. PC11                            0.01945  ██████████████████████████████████████
   7. PC19                            0.01893  █████████████████████████████████████
   8. PC42                            0.01866  █████████████████████████████████████
   9. PC28                            0.01825  ████████████████████████████████████
  10. PC50                            0.01818  ████████████████████████████████████
  11.

## 12d. 各 ddg 间相关性矩阵

检查四种 ddg 之间的相关性，理解冗余/互补的来源。

In [5]:
from scipy.stats import spearmanr

ddg_names = ["ddg_esm2", "ddg_struct", "ddg_rasp", "ddg_foldx"]
ddg_mat = np.column_stack([ddg_sources[n].ravel() for n in ddg_names])

# 成对 Spearman
print(f"\n{'='*70}")
print(f"  ddg 间 Spearman 秩相关矩阵")
print(f"  {'':>14s}", end="")
for n in ddg_names:
    print(f"{n:>12s}", end="")
print()
for i, ni in enumerate(ddg_names):
    print(f"  {ni:>14s}", end="")
    for j, nj in enumerate(ddg_names):
        mask = np.isfinite(ddg_mat[:, i]) & np.isfinite(ddg_mat[:, j])
        r, _ = spearmanr(ddg_mat[mask, i], ddg_mat[mask, j])
        print(f"{r:>12.4f}", end="")
    print()

print(f"\n  解读:")
print(f"  - |r| > 0.6: 高度冗余，合在一起可能不会带来额外增益")
print(f"  - |r| < 0.3: 几乎正交，互补性强")
print(f"  - 0.3 ≤ |r| ≤ 0.6: 中等相关，部分互补")
print(f"{'='*70}")


  ddg 间 Spearman 秩相关矩阵
                    ddg_esm2  ddg_struct    ddg_rasp   ddg_foldx
        ddg_esm2      1.0000      0.1445      0.4404      0.4491
      ddg_struct      0.1445      1.0000      0.2540      0.1410
        ddg_rasp      0.4404      0.2540      1.0000      0.6801
       ddg_foldx      0.4491      0.1410      0.6801      1.0000

  解读:
  - |r| > 0.6: 高度冗余，合在一起可能不会带来额外增益
  - |r| < 0.3: 几乎正交，互补性强
  - 0.3 ≤ |r| ≤ 0.6: 中等相关，部分互补
